This Notebook is do data preprocessing and Machine Learning Model traning on data.
Below is the code that load libraries

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib

dataFile = pd.read_csv("healthcare-dataset-stroke-data.csv")
dataFile.head()


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


Task 1, Statistical Analysis

In [10]:
dataFile.describe()

,id,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
count,5110.000000,5110.000000,5110.000000,5110.000000,5110.000000,4909.000000,5110.000000
mean,36517.829354,43.226614,0.097456,0.054012,106.147677,28.893237,0.048728
std,21161.721625,22.612647,0.296607,0.226063,45.283560,7.854067,0.215320
min,67.000000,0.080000,0.000000,0.000000,55.120000,10.300000,0.000000
25%,17741.250000,25.000000,0.000000,0.000000,77.245000,23.500000,0.000000
50%,36932.000000,45.000000,0.000000,0.000000,91.885000,28.100000,0.000000
75%,54682.000000,61.000000,0.000000,0.000000,114.090000,33.100000,0.000000
max,72940.000000,82.000000,1.000000,1.000000,271.740000,97.600000,1.000000


Checked The data types of all the coulmns of CSV file

In [13]:
dataFile.dtypes

id                     int64
gender                object
age                  float64
hypertension           int64
heart_disease          int64
ever_married          object
work_type             object
Residence_type        object
avg_glucose_level    float64
bmi                  float64
smoking_status        object
stroke                 int64
dtype: object

Checking the correlation of columns with each other

In [53]:
# Numeric columns correlation
dataFile.corr(numeric_only=True)

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,id,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
Unnamed: 0.2,1.000000,1.000000,1.000000,0.014814,-0.107892,-0.060343,-0.067657,-0.068993,-0.044122,-0.372908
Unnamed: 0.1,1.000000,1.000000,1.000000,0.014814,-0.107892,-0.060343,-0.067657,-0.068993,-0.044122,-0.372908
Unnamed: 0,1.000000,1.000000,1.000000,0.014814,-0.107892,-0.060343,-0.067657,-0.068993,-0.044122,-0.372908
id,0.014814,0.014814,0.014814,1.000000,0.003538,0.003550,-0.001296,0.001092,0.002999,0.006388
age,-0.107892,-0.107892,-0.107892,0.003538,1.000000,0.276398,0.263796,0.238171,0.325942,0.245257
hypertension,-0.060343,-0.060343,-0.060343,0.003550,0.276398,1.000000,0.108306,0.174474,0.160189,0.127904
heart_disease,-0.067657,-0.067657,-0.067657,-0.001296,0.263796,0.108306,1.000000,0.161857,0.038899,0.134914
avg_glucose_level,-0.068993,-0.068993,-0.068993,0.001092,0.238171,0.174474,0.161857,1.000000,0.168751,0.131945
bmi,-0.044122,-0.044122,-0.044122,0.002999,0.325942,0.160189,0.038899,0.168751,1.000000,0.038947
stroke,-0.372908,-0.372908,-0.372908,0.006388,0.245257,0.127904,0.134914,0.131945,0.038947,1.000000


Checking the Null values

In [54]:
dataFile.isna().sum()

Unnamed: 0.2         0
Unnamed: 0.1         0
Unnamed: 0           0
id                   0
gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
Residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
stroke               0
dtype: int64

There is only one column that have missing values is 'bmi'
So, checks the medain and mean of that columns and fill missing values with approperate one

In [55]:

dataFile['bmi'].median()


np.float64(28.4)

Mean of the 'bmi' column

In [56]:
dataFile['bmi'].mean()

np.float64(28.893236911794663)

Checks the standard deviation of the columns for checking that how much values are dipersed from average value.

In [57]:
dataFile['bmi'].std()

np.float64(7.698017826857082)

Checks the varraince of 'bmi' column

In [58]:
dataFile['bmi'].var()

np.float64(59.25947846260943)

So, after statistical analysis we checked that, bmi column has not too much dispersed values, so we decide to fill the missing values with mean of the column and saved the preproceesed data into new CSV file.

In [59]:
dataFile['bmi']=dataFile['bmi'].fillna(dataFile['bmi'].mean())
dataFile.to_csv("ProcessedCSVStrokData.csv", index=False)


Now, assign newly created CSV file, that dont has any missing values.

In [9]:
dataFile=pd.read_csv('ProcessedCSVStrokData.csv')
dataFile.isna().sum()

id                   0
gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
Residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
stroke               0
dtype: int64

Score Normalization
age, avg_glucose_level and bmi column will be normalized by the formula below
newValue=(oldValue-minimumValue)/(maximimValue-minimumValue)

In [ ]:
dataFile['age']=(dataFile['age']-dataFile['age'].min())/(dataFile['age'].max()-dataFile['age'].min())
dataFile['bmi']=(dataFile['bmi']-dataFile['bmi'].min())/(dataFile['bmi'].max()-dataFile['bmi'].min())
dataFile['avg_glucose_level']=(dataFile['avg_glucose_level']-dataFile['avg_glucose_level'].min())/(dataFile['avg_glucose_level'].max()-dataFile['avg_glucose_level'].min())
dataFile.to_csv('NormalizedStrokeData.csv', index=False)


In [10]:
dataFile.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.600000,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,28.893237,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.500000,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.400000,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.000000,never smoked,1


Data Encoding Phase
Gender, Residence_type and smoking_status are need to be encoded for training of machine learning model effecively.

In [20]:
dataFile['gender_map']=dataFile['gender'].map({
    "Female" :0 ,
    "Male":1,
    "Other":2
})


In [21]:
dataFile['ever_married_map']=dataFile['ever_married'].map({
    "No":0 ,
    "Yes":1
})


In [22]:
dataFile['work_type_map']=dataFile['work_type'].map({
    "children":0,
    "Govt_job":1,
    "Never_worked":2,
    "Private":3,
    "Self-employed":4
})

In [ ]:
dataFile['Residence_type_map']=dataFile['Residence_type'].map({
    "Rural":0,
    "Urban":1
})

In [25]:
dataFile['smoking_status_map']=dataFile['smoking_status'].map({
    "never smoked":0,
    "formerly smoked":1,
    "smokes":2,
    "Unknown":3,
})

In [26]:
dataFile.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke,gender_map,ever_married_map,work_type_map,Residence_type_map,smoking_status_map
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.600000,formerly smoked,1,1,1,3,1,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,28.893237,never smoked,1,0,1,4,0,0
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.500000,never smoked,1,1,1,3,0,0
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.400000,smokes,1,0,1,3,1,2
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.000000,never smoked,1,0,1,4,0,0


Data Preprocessing Is complete, now save the file and display the head

In [ ]:
dataFile.to_csv('NormalizedStrokeData.csv', index=False)
dataFile.head()